# Member 1 EDA Notebook: NYC Street Tree Health Predictor

**Role:** Member 1 — Data and EDA Lead  
**Project:** NYC Street Tree Health Predictor  
**Goal:** Prepare a clean dataset, create exploratory visualizations, and document insights for the final Streamlit app.

This notebook is designed as a clean handoff to the modeling and app teammates. It uses the Member 1 cleaned dataset included in this package.

## 1. Imports and data loading

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("data/nyc_tree_member1_clean_sample.csv")
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 2. Dataset overview

The target variable is `health`, with classes **Good**, **Fair**, and **Poor**. The most important Member 1 engineered features are:

- `problem_count`
- `has_problem`
- `dbh_group`
- `species_top15_or_other`
- binary root/trunk/branch problem flags ending in `_flag`

In [ ]:
df.info()

df[['health','tree_dbh','spc_common','boroname','problem_count','has_problem','dbh_group']].describe(include='all')

## 3. Target distribution

This chart is important because it shows class imbalance. Member 2 should use **macro-F1** and a **confusion matrix**, not only accuracy.

In [ ]:
health_counts = df['health'].value_counts().reindex(['Good','Fair','Poor'])
ax = health_counts.plot(kind='bar', color=['#2E7D32','#F9A825','#C62828'], figsize=(7,4), title='Tree Health Distribution')
ax.set_xlabel('Health')
ax.set_ylabel('Number of trees')
for i, v in enumerate(health_counts):
    ax.text(i, v, f'{v:,}
{v/len(df):.1%}', ha='center', va='bottom')
plt.tight_layout()
plt.show()
health_counts / len(df)

## 4. Borough patterns

This section supports the Data Visualization app page. The chart compares the health mix by borough.

In [ ]:
borough_health = pd.crosstab(df['boroname'], df['health'], normalize='index') * 100
borough_health = borough_health[[c for c in ['Good','Fair','Poor'] if c in borough_health.columns]]
ax = borough_health.plot(kind='bar', stacked=True, figsize=(9,5), color=['#2E7D32','#F9A825','#C62828'], title='Health Mix by Borough (%)')
ax.set_ylabel('Percent')
ax.set_xlabel('Borough')
plt.legend(title='Health', bbox_to_anchor=(1.02,1), loc='upper left')
plt.tight_layout()
plt.show()
borough_health.round(2)

## 5. Species patterns

The raw species field has many categories, so Member 1 created `species_top15_or_other` for easier modeling and cleaner dropdowns.

In [ ]:
top_species = df['spc_common'].value_counts().head(12)
ax = top_species.sort_values().plot(kind='barh', figsize=(8,5), color='#388E3C', title='Top 12 Species')
ax.set_xlabel('Number of trees')
plt.tight_layout()
plt.show()

top_species

## 6. Visible problem fields

The original root/trunk/branch fields were turned into binary flags and summarized with `problem_count` and `has_problem`. These are good app features because they are understandable to users.

In [ ]:
problem_health = pd.crosstab(df['has_problem'], df['health'], normalize='index') * 100
problem_health = problem_health[[c for c in ['Good','Fair','Poor'] if c in problem_health.columns]]
ax = problem_health.plot(kind='bar', stacked=True, figsize=(7,4), color=['#2E7D32','#F9A825','#C62828'], title='Health Mix by Problem Status (%)')
ax.set_ylabel('Percent')
ax.set_xlabel('Has any recorded problem?')
plt.legend(title='Health', bbox_to_anchor=(1.02,1), loc='upper left')
plt.tight_layout()
plt.show()
problem_health.round(2)

## 7. Diameter groups

`tree_dbh` is numeric, while `dbh_group` makes the app easier to read. Both can be useful: `tree_dbh` for modeling and `dbh_group` for visualization.

In [ ]:
dbh_health = pd.crosstab(df['dbh_group'], df['health'], normalize='index') * 100
dbh_health = dbh_health[[c for c in ['Good','Fair','Poor'] if c in dbh_health.columns]]
ax = dbh_health.plot(kind='bar', stacked=True, figsize=(9,5), color=['#2E7D32','#F9A825','#C62828'], title='Health Mix by Diameter Group (%)')
ax.set_ylabel('Percent')
ax.set_xlabel('Diameter group')
plt.xticks(rotation=25, ha='right')
plt.legend(title='Health', bbox_to_anchor=(1.02,1), loc='upper left')
plt.tight_layout()
plt.show()
dbh_health.round(2)

## 8. Data limitations

Use these points in the Streamlit app and presentation:

- The target classes are imbalanced, with Good usually the majority class.
- The data comes from the 2015 census, so this is not a live monitoring tool.
- The model can show associations, not proven causes.
- Visible problem fields may reflect reporting or inspection bias.
- The app should be framed as an educational/triage tool, not an official maintenance decision system.

## 9. Handoff recommendation

For Member 2, start with this feature set:

- Numeric: `tree_dbh`, `problem_count`
- Categorical: `species_top15_or_other`, `steward`, `guards`, `sidewalk`, `has_problem`, `boroname`, `dbh_group`, `curb_loc`
- Target: `health`

For Member 3, copy the `data/` folder and the two Streamlit page files into the final app repository.